# J-lens / Antidoom baseline — `LiquidAI/LFM2-2.6B`

**Not** the blog's private early LFM2.5-2.6B. **Not** Antidoom-trained.

| Setting | Value |
|---|---|
| Model | `LiquidAI/LFM2-2.6B` |
| Thinking | leave **default** (dynamic hybrid reasoning — do not force-disable) |
| Prompts | 200 stratified antidoom-mix reasoning sources, seed=42 |
| max_new_tokens | **4000** |
| temperature | 0.01 |
| Backend | **vLLM** preferred (`fp8` → fallback `bfloat16`) |

Connect this notebook to a **Google Colab GPU kernel** (Colab extension), or Runtime→GPU in browser Colab.

In [ ]:
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# PICK ONE setup method (Colab VM does NOT see your laptop files directly):
#
# A) GITHUB (recommended): push repo once, paste URL below, leave DRIVE_ZIP=""
# B) DRIVE ZIP: zip the whole j-lens folder → upload to My Drive → set path below
# C) MANUAL: if you know the path on the VM, set PROJECT_OVERRIDE
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/Mithilyaganti/jlens-doom-loop-analysis.git"
DRIVE_ZIP = ""
PROJECT_OVERRIDE = ""  # e.g. "/content/j-lens" — only if you already cloned/unzipped

def find_repo_root() -> Path | None:
    if PROJECT_OVERRIDE:
        p = Path(PROJECT_OVERRIDE)
        return p if (p / "jspace" / "model_config.py").is_file() else None
    # Colab VM: repo often lands in /content after clone/unzip
    for candidate in [Path.cwd(), *Path.cwd().parents, Path("/content/j-lens"), Path("/content")]:
        if (candidate / "jspace" / "model_config.py").is_file():
            return candidate
        for sub in candidate.glob("*/jspace/model_config.py"):
            return sub.parent
    return None

PROJECT = find_repo_root()
if PROJECT is None:
    if REPO_URL:
        get_ipython().system(f"git clone --depth 1 {REPO_URL} /content/j-lens")
        PROJECT = Path("/content/j-lens")
    elif DRIVE_ZIP and Path(DRIVE_ZIP).is_file():
        from google.colab import drive
        drive.mount("/content/drive")
        get_ipython().system(f'unzip -q -o "{DRIVE_ZIP}" -d /content')
        PROJECT = find_repo_root() or Path("/content/j-lens")
    else:
        raise RuntimeError(
            "Colab cannot see your laptop folder. Do ONE of:\n"
            "  1) Push to GitHub → set REPO_URL above → re-run this cell\n"
            "  2) Zip j-lens → upload to Drive → set DRIVE_ZIP → re-run\n"
            "  (GitHub is faster to set up than Drive zip.)"
        )

if not (PROJECT / "jspace").is_dir():
    raise RuntimeError(f"Repo incomplete at {PROJECT}")

os.chdir(PROJECT)
print("PROJECT", PROJECT.resolve())
print("prompt sample", (PROJECT / "results" / "prompt_sample_ids.json").is_file())

In [ ]:
# Install deps + vendored jlens (not in GitHub clone — fetched at runtime)
%pip install -q -U "pandas>=2.1,<2.4" transformers accelerate bitsandbytes datasets huggingface_hub \
    scipy statsmodels tqdm pyyaml safetensors sentencepiece matplotlib seaborn

try:
    %pip install -q vllm
    print("vllm ok")
except Exception as e:
    print("vllm install failed — will use HF:", e)

# Clone + install open-jlens if missing (fixes ModuleNotFoundError: jlens)
!python scripts/00_colab_vendors.py

import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
import os
from pathlib import Path

os.environ["JLENS_MODEL"] = "LiquidAI/LFM2-2.6B"
os.environ["JLENS_BASELINE_PROMPTS"] = "200"
os.environ["JLENS_SAMPLE_SEED"] = "42"
os.environ["JLENS_MAX_NEW_TOKENS"] = "4000"
os.environ["JLENS_TEMPERATURE"] = "0.01"
os.environ["JLENS_BACKEND"] = "vllm"          # or "hf"
os.environ["JLENS_VLLM_DTYPE"] = "fp8"         # falls back to bfloat16 inside wrapper
os.environ["JLENS_MAX_MODEL_LEN"] = "6000"
os.environ["JLENS_HF_QUANTIZE"] = "0"          # on Colab prefer bf16/fp8 via vLLM
os.environ["PYTHONPATH"] = str(Path.cwd())

DRIVE_RESULTS = Path("/content/drive/MyDrive/j-lens-results-lfm2-2.6b")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print(dict((k, os.environ[k]) for k in os.environ if k.startswith("JLENS_")))

In [ ]:
# Full LFM2-2.6B protocol (resumes from checkpoint if interrupted)
# - 06_fit_lfm_lens.py: status stub (Exp1–3 blocked until lens exists)
# - 02_baseline_pass.py: 200 prompts, max_new=4000
# - write_lfm_report.py: honest RUN_REPORT
!python scripts/run_lfm2_26b.py

In [ ]:
import json
from pathlib import Path

summary = Path("results/baseline_pass_summary.json")
report = Path("results/RUN_REPORT_lfm2-2.6b_antidoom_mix_200.md")
ckpt = Path("results/checkpoints/baseline_pass_lfm2-2.6b.json")

if ckpt.is_file():
    n = len(json.loads(ckpt.read_text())["completed_prompt_ids"])
    print(f"checkpoint progress: {n}/200")
if summary.is_file():
    s = json.loads(summary.read_text())
    print(f"loops: {s.get('n_loop')} rate={s.get('loop_rate', 0):.1%} backend={s.get('backend')}")
if report.is_file():
    print("\n--- REPORT ---\n")
    print(report.read_text())
else:
    !python scripts/write_lfm_report.py
    if report.is_file():
        print(report.read_text())

## After baseline: J-lens fit → Exp1–3

Only after a real lens is saved to `lenses/lfm2-2.6b.pt`:

```bash
python scripts/01_workspace_band.py
python scripts/03_exp1_static_geometry.py
python scripts/04_exp2_dynamic.py
python scripts/05_exp3_causal.py
```

Update those scripts' hard-coded `qwen3.5-4b` artifact names to use `jspace.model_config.artifact_paths()` if not already done.